# Mini-Projeto Avaliativo - Módulo1 - Semana 07

#### Seja bem vindo ao meu Mini-Projeto Avaliativo: O objetivo deste projeto é realizar uma Análise Exploratória de Dados (AED) sobre a base de varejo, verificando a qualidade dos dados, realizando limpeza, criando estatística descritiva e agrupamentos com enfâse em gerar insights valiosos para possíveis tomadas de decisões no varejo.

# Importação e Inspeçao Visual dos Dados

### Importando biblioteca Pandas

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

### Carregamento da base Varejo com pandas

In [3]:
base_varejo = "Base Varejo.csv"


df = pd.read_csv(base_varejo, sep=";", encoding="utf-8-sig")

# Mostrando as primeiras 5 linhas
df.head()

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


### VISÃO GERAL DO DATASET

In [4]:
# Já vou limpars os espaços para ter uma visualização melhor
df.columns = df.columns.str.strip()

# E remover as colunas sobressalentes criadas pelo ";"
df = df.loc[:, ~df.columns.str.contains('^Unnamed', regex=True)]

In [5]:
# Quantidade de linhas e colunas
df_Inicial = df.shape
df_Inicial

(830000, 10)

In [6]:
# Tipos por coluna
df.dtypes

DATA           str
CO_ID        int64
CL_ID        int64
CL_GENERO      str
CL_EC        int64
CL_FHL       int64
CL_SEG         str
PR_ID        int64
PR_CAT         str
PR_NOME        str
dtype: object

In [7]:
# Contando as duplicatas
print(f"\nTotal de linhas duplicadas: {df.duplicated().sum()}")


Total de linhas duplicadas: 96553


In [8]:
# Mostra as duplicadas ordenadas por ID de Compra e Produto para comparar lado a lado
duplicadas_ordenadas = df[df.duplicated(keep=False)].sort_values(by=['CO_ID', 'PR_ID'])

print("--- DUPLICADAS ORDENADAS LADO A LADO ---")
print(duplicadas_ordenadas.head(10))

--- DUPLICADAS ORDENADAS LADO A LADO ---
          DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID  \
3   01/02/2019   1000    534         M      4       1      C      4   
40  01/02/2019   1000    534         M      4       1      C      4   
7   01/02/2019   1000    534         M      4       1      C     11   
46  01/02/2019   1000    534         M      4       1      C     11   
14  01/02/2019   1000    534         M      4       1      C     13   
19  01/02/2019   1000    534         M      4       1      C     13   
22  01/02/2019   1000    534         M      4       1      C     69   
49  01/02/2019   1000    534         M      4       1      C     69   
15  01/02/2019   1000    534         M      4       1      C    218   
50  01/02/2019   1000    534         M      4       1      C    218   

       PR_CAT             PR_NOME  
3   ALIMENTOS             ABACAXI  
40  ALIMENTOS             ABACAXI  
7   ALIMENTOS              AZEITE  
46  ALIMENTOS              AZEITE

In [9]:
# Verificando os valores nulos
df.isnull().sum()

DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64

#### Verificando se há algum valor incompleto

In [10]:
df[df.isnull().any(axis=1)]

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME


# Tratando as divergências encontradas

#### Conversão de datas

In [11]:
# Convertendo coluna de data para Datetime
if "DATA" in df.columns:
    df["DATA"] = pd.to_datetime(df["DATA"], format='%d/%m/%Y', errors='coerce')

#### Revisitando os tipos de coluna para validar a transformação da coluna DATA

In [12]:
df.dtypes

DATA         datetime64[us]
CO_ID                 int64
CL_ID                 int64
CL_GENERO               str
CL_EC                 int64
CL_FHL                int64
CL_SEG                  str
PR_ID                 int64
PR_CAT                  str
PR_NOME                 str
dtype: object

#### Caso haja uma linha na coluna (PR_CAT) com campo nulo o código abaixo tratara isto substituindo por "Sem Categoria"
#### Caso haja uma linha na coluna (PR_NOME) com campo nulo o código abaixo tratara isto substituindo por "Não Informado"

In [13]:
# Substitui '#N/D', textos vazios ou espaços por NaN em todo o DataFrame
df = df.replace(["#N/D", "#N/A", "#ND", ""], np.nan)
df = df.replace(r"^\s*$", np.nan, regex=True)

# Depois preenche as colunas com os textos padrão desejados
if "PR_CAT" in df.columns:
    df["PR_CAT"] = df["PR_CAT"].fillna("Sem Categoria")
if "PR_NOME" in df.columns:
    df["PR_NOME"] = df["PR_NOME"].fillna("Não Informado")



#### Padronização dos textos das colunas com tipos de dados Strings

In [14]:
df["CL_GENERO"] = df["CL_GENERO"].str.title()
df["CL_SEG"] = df["CL_SEG"].str.title()
df["PR_CAT"] = df["PR_CAT"].str.title()
df["PR_NOME"] = df["PR_NOME"].str.title()

#### Remoção de duplicatas

In [15]:
df = df.drop_duplicates()

# Conferindo se zerou as duplicatas
print(f"\nTotal de linhas duplicadas depois da remoção: {df.duplicated().sum()}")


Total de linhas duplicadas depois da remoção: 0


In [16]:
# Conferindo a quantidade após a remoção de duplicatas
df.shape

(733447, 10)

# Estatisticas Básicas da Coluna N° de Filhos (CL_FHL)

In [17]:
# Gerando todas as estatísticas padrões do describe apenas para CL_FHL
desc = df["CL_FHL"].describe()

# Exibindo os resultados e adicionando a Moda
print("--- ESTATÍSTICA DESCRITIVA DA COLUNA NÚMERO DE FILHOS (CL_FHL) ---")
print(f"Contagem (N):     {desc['count']}")
print(f"Média:            {desc['mean']:.2f}")
print(f"Desvio Padrão:    {desc['std']:.2f}")
print(f"Mínimo:           {desc['min']}")
print(f"1º Quartil (25%): {desc['25%']}")
print(f"Mediana (50%):    {desc['50%']}")
print(f"3º Quartil (75%): {desc['75%']}")
print(f"Máximo:           {desc['max']}")
print(f"Moda:             {df['CL_FHL'].mode()[0]}") # Moda calculada separadamente

--- ESTATÍSTICA DESCRITIVA DA COLUNA NÚMERO DE FILHOS (CL_FHL) ---
Contagem (N):     733447.0
Média:            1.15
Desvio Padrão:    1.42
Mínimo:           0.0
1º Quartil (25%): 0.0
Mediana (50%):    0.0
3º Quartil (75%): 2.0
Máximo:           4.0
Moda:             0


# Agrupamentos e Padrões utilizando do GROUPBY

#### Consumo por Gênero

In [18]:
analise_genero = df.groupby("CL_GENERO")["CO_ID"].agg(
    Qtd_Compras="nunique",      # Quantidade de pedidos/notas
    Total_Itens="count"          # Total de itens nas notas
)

# Calcula a média de itens que cada gênero leva por compra
analise_genero["Itens_Por_Compra"] = (analise_genero["Total_Itens"] / analise_genero["Qtd_Compras"]).round(2)

print(analise_genero)

           Qtd_Compras  Total_Itens  Itens_Por_Compra
CL_GENERO                                            
F                 9615       382427             39.77
M                 8856       351020             39.64


#### Produtos mais comprados por mulheres

In [19]:
# Filtrando o público feminino
df_mulheres = df[df["CL_GENERO"] == "F"]

prod_mulheres = (
    df_mulheres.groupby("PR_NOME")["CO_ID"]
    .count()
    .reset_index(name="Qtd_Comprada")
    .sort_values(by="Qtd_Comprada", ascending=False)
    .head(5)
)

print("\n--- Top 5 Produtos mais comprados por Mulheres ---")
print(prod_mulheres)


--- Top 5 Produtos mais comprados por Mulheres ---
             PR_NOME  Qtd_Comprada
85   Presunto Cozido          6613
27           Chupeta          3485
108         Sardinha          3472
98         Removedor          3452
37        Detergente          3443


#### Categorias mais consumidas de acordo com a Classe Social

In [20]:
# Agrupando por Classe Social (CL_SEG) e Categoria (PR_CAT)
consumo_classe = (
    df.groupby(["CL_SEG", "PR_CAT"])["CO_ID"]
    .count()
    .reset_index(name="Total_Itens")
    .sort_values(by=["CL_SEG", "Total_Itens"], ascending=[True, False])
)

print("--- Preferência de Categoria por Classe Social ---")
print(consumo_classe)

--- Preferência de Categoria por Classe Social ---
   CL_SEG         PR_CAT  Total_Itens
1       A      Alimentos        31118
3       A        Higiene        11326
4       A        Limpeza        10460
2       A        Bebidas         3166
5       A            Pet         2303
0       A     Acessorios         1043
6       A  Sem Categoria          261
8       B      Alimentos       245501
10      B        Higiene        87943
11      B        Limpeza        82063
9       B        Bebidas        24363
12      B            Pet        18406
7       B     Acessorios         8165
13      B  Sem Categoria         2064
15      C      Alimentos       107578
17      C        Higiene        38433
18      C        Limpeza        36109
16      C        Bebidas        10735
19      C            Pet         7844
14      C     Acessorios         3663
20      C  Sem Categoria          903


#### Consumo de acordo com N° de filhos e Categoria de produtos

In [21]:
# Agrupando por Número de Filhos e Categoria do Produto
consumo_filhos = (
    df.groupby(["CL_FHL", "PR_CAT"])["CO_ID"]
    .count()
    .reset_index(name="Qtd_Itens")
    .sort_values(by=["CL_FHL", "Qtd_Itens"], ascending=[True, False])
)

print("--- Categorias mais compradas de acordo com o N° de Filhos ---")
print(consumo_filhos)

--- Categorias mais compradas de acordo com o N° de Filhos ---
    CL_FHL         PR_CAT  Qtd_Itens
1        0      Alimentos     201670
3        0        Higiene      72302
4        0        Limpeza      67625
2        0        Bebidas      19991
5        0            Pet      14960
0        0     Acessorios       6755
6        0  Sem Categoria       1683
8        1      Alimentos      47613
10       1        Higiene      17018
11       1        Limpeza      15782
9        1        Bebidas       4901
12       1            Pet       3561
7        1     Acessorios       1564
13       1  Sem Categoria        406
15       2      Alimentos      49269
17       2        Higiene      17647
18       2        Limpeza      16583
16       2        Bebidas       4895
19       2            Pet       3721
14       2     Acessorios       1647
20       2  Sem Categoria        406
22       3      Alimentos      48284
24       3        Higiene      17487
25       3        Limpeza      16148
23       3  

# Exportando a Base Limpa

In [22]:
df.to_csv(
    "base_varejo_limpa.csv", index=False
)

# Bloco de Conclusões

In [30]:
total_linhas_iniciais = df_Inicial[0]
total_linhas_finais = df.shape[0]
total_duplicadas_removidas = total_linhas_iniciais - total_linhas_finais
nulos_categoria_tratados = (df['PR_CAT'] == "Sem Categoria").sum()
data_minima = df['DATA'].min().strftime('%d/%m/%Y') if 'DATA' in df.columns else "N/A"
data_maxima = df['DATA'].max().strftime('%d/%m/%Y') if 'DATA' in df.columns else "N/A"

In [31]:
print("\n" + "="*60)
print("           RELATÓRIO FINAL DE QUALIDADE E EXECUÇÃO (ETL)          ")
print("="*60)
print(f"• Total de registros válidos (Iniciais)     : {total_linhas_iniciais}")
print(f"• Total de registros válidos (Finais)     : {total_linhas_finais}")
print(f"• Linhas duplicadas eliminadas             : {total_duplicadas_removidas}")
print(f"• Categorias vazias/ajustadas ('Sem Cat')  : {nulos_categoria_tratados}")
print(f"• Período das vendas analisadas            : {data_minima} até {data_maxima}")
print("-" * 60)
print("RESUMO RÁPIDO - NÚMERO DE FILHOS (CL_FHL):")
print(f"• Média: {df['CL_FHL'].mean():.2f} | Mediana: {df['CL_FHL'].median():.1f} | Moda: {df['CL_FHL'].mode()[0]}")
print("="*60 + "\n")


           RELATÓRIO FINAL DE QUALIDADE E EXECUÇÃO (ETL)          
• Total de registros válidos (Iniciais)     : 830000
• Total de registros válidos (Finais)     : 733447
• Linhas duplicadas eliminadas             : 96553
• Categorias vazias/ajustadas ('Sem Cat')  : 3228
• Período das vendas analisadas            : 04/01/2019 até 08/12/2022
------------------------------------------------------------
RESUMO RÁPIDO - NÚMERO DE FILHOS (CL_FHL):
• Média: 1.15 | Mediana: 0.0 | Moda: 0



#### Insights

As Mulheres mostraram um volume superior de compras quando comparamos com os homens, a quantidade de compras a mais é de 759 compras e o consumo de itens 31.407 a mais que os homens. Foi feito o levantamento dos produtos mais consumidos por elas, podemos assim estruturar o Varejo de forma que produtos menos vendidos fiquem mais próximos daqueles que as mulherer mais procuram, para ter mais visualização para eles.

Verificando as classes sociais vemos que o foco maior de todas está na categoria alimentos(com aproximadamente 50%), seguido por higiêne (cerca de 18%) e limpeza (cerca de 17%).

Clientes sem filhos concentram o maior volume absoluto de compras nesta rede, somando aproximadamente 385 mil itens vendidos (Eles possuem um pouco mais de 50% do total da base). Em seguida estão as famílias que possue entre 1 a 3 filhos, já quanto as familias com 4 filhos é aprensentado uma queda grande no poder de consumo.

#### Possíveis problemas remanescentes

Ainda podemoríamos verificar o consumo obtido pelos compradores de acordo com seu estado civil. Estudar o consumos de pais ou mães solteiras e ou viúvas.
Diversos Insights poderiam surgir desta base.

Sinceramente creio ter feito todas as limpezas que consegui verificar, como: modificação de data, identificação de Inteiros, Floats, Strings, ajuste dos valores nulos, eliminação de duplicatas e igualar os textos padronizando-os usando str.title().

Espero que tenha gostado, obrigado!